<a href="https://colab.research.google.com/github/franciscoweitzman-cyber/aps_francisco_weitzman/blob/main/TS5/TS5_WEITZMAN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Trabajo Semanal N° 5
## Por: Francisco Weitzman
### UNSAM


## Introducción
En la cotideaneidad trabajamos con señales reales, las que trabajamos día a día como lo pueden ser un audio de Whatsapp, un archivo .wav de música, o aplicaciones más técnicas como el análisis de un Electrocardiograma o un Electroencefalograma, señales de radio del espacio exterior o apuntando nuestros instrumentos para el otro lado midiendo señales sísmicas de nuestro planeta. Todas estas comparten una característica, y es que no están en el plano determinista, sino que son sistemas probabilisticos. Esto se debe a que ni las fuentes emisoras de señal son perfectas, ni el medio en el cual se propagan, ni nuestros instrumentos de medición. Esto es, no son Lineales como veníamos asumiendo hasta ahora. En un sistema lineal, una perturbación resultante se puede entender como la suma de dos perturbaciones que la generaron, y si se conocen las perturbaciones que la iniciaron, se puede estudiar la resultante analizando las perturbaciones individuales por separado. Acá estamos teniendo en cuenta que si yo perturbo a un sistema, no va a responder linealmente. Por otro lado, el ruido aportado por cada eslabón en la vida de una señal también tenemos que poder distinguir de alguna manera. Así como nuestro cerebro puede distinguir (de manera increíble para mí) lo que me está diciendo una persona en un aula llena de otros diálogos y otros ruidos, o cómo podemos distinguir distintos instrumentos en una canción que escuchamos por el celular o en el Teatro Colón a 20m de distancia, deberíamos ser capaces con nuestra tecnología de separar las señales que son de nuestro interés, y así es como distinguimos por ejemplo un descubrimiento científico de una simple medición.

Para eso existe el Periodograma, que es una estimación espectral de potencia (PSD por sus siglas en inglés). Este se define como:

$S_{xx}(ω,N)=\frac{1}{N}|\sum_{0}^{N-1}X[n]*e^{-jωn}|^2$


Vemos que se consigue a través de la DTFT. Ahora bien, si bien se considera que este es el "verdadero" estimador de nuestro PSD, no es un gran estimador ya que no mejora la varianza a medida que aumenta la cantidad de muestras, aunque sea asintóticamente insesgado, por lo que no es un gran estimador. Por eso, hice uso del método de Welch.

Este consiste en promediar periodogramas con el fin de reducir su varianza. La forma en que lo logra es mediante la toma de M segmentos de tamaño L de muestras y de un solapamiento D. Su fórmula para tiempo discreto es:

$P_W[k, L, D, M] = \left. P_W(\omega, L, D, M) \right|_{\omega = 2\pi k / L}=$

$
= \frac{\frac{1}{M} \sum_{m=0}^{M-1} \left( \left| \sum_{n=0}^{L-1} x[n+mD] w_L[n] e^{-j2\pi kn/L} \right|^2 \right)}{\sum_{n=0}^{L-1} w_L^2[n]}
 $

 $= \frac{\frac{1}{M} \sum_{m=0}^{M-1} \left( \left| \text{DFT}_L (x[n+mD] w_L[n]) \right|^2 \right)}{\sum_{n=0}^{L-1} w_L^2[n]}
$

Si se elije un solapamiendo D = L/2, la varianza se puede reducir hasta un 50% con respecto al periodograma. Si se aumenta la cantidad de segmentos, se mejora la resolución porque hay menos promediado, pero por esta misma razón la varianza aumenta. El objetivo es conseguir un criterio para reducir la varianza sin perder información de nuestra señal.


En el desarrollo voy a usar las funciones de la librería scipy.

## Desarrollo
Trabajamos con el ECG sin ruido:

In [10]:
"""
Created on Wed Sep 30 11:22:08 2026

@author: Fran
"""
import numpy as np
from scipy import signal as sig

import matplotlib.pyplot as plt
import pandas as pd
import scipy.io as sio
# from scipy.io.wavfile import
##################
## ECG sin ruido
##################
fs_ecg = 1000 # Hz
ecg_one_lead = np.load('ecg_sin_ruido.npy')
tiempo_ecg = np.arange(0,len(ecg_one_lead))/fs_ecg

var_ECG = np.var(ecg_one_lead) # Energia de la señal en el tiempo
f_ecg, PDS_ecg = sig.welch(ecg_one_lead, fs_ecg, nperseg = len(ecg_one_lead)/2)
f_ecg2, PDS_ecg2 = sig.welch(ecg_one_lead, fs_ecg, nperseg = len(ecg_one_lead)/10)

pds_ecg_db = 10* (np.log10(PDS_ecg))
pds_ecg2_db = 10* (np.log10(PDS_ecg2))

# Análisis de Energía
df = f_ecg2[2]-f_ecg2[1]
en_pds_ecg2 = np.sum(PDS_ecg2)*df
print(en_pds_ecg2/var_ECG)

plt.figure()
plt.title("ECG Sin Ruido")
plt.xlabel('time [s]')
plt.ylabel('Amplitud [V]')
plt.plot(tiempo_ecg[5000:15000], ecg_one_lead[5000:15000])
plt.show()

plt.figure()
plt.title("ECG Sin Ruido [dB(Hz)]")
plt.xlabel('frequency [Hz]')
plt.ylabel('PSD [dB]')
plt.plot(f_ecg, pds_ecg_db, label = '1')
plt.plot(f_ecg2, pds_ecg2_db, label = '2')
plt.legend(loc='upper right')
plt.show()

plt.figure()
plt.title("ECG Sin Ruido [dB(Hz)]")
plt.xlabel('frequency [Hz]')
plt.ylabel('PSD [dB]')
plt.plot(f_ecg[:len(f_ecg)//10], pds_ecg_db[:len(f_ecg)//10], label = '1')
plt.plot(f_ecg2[:len(f_ecg2)//10], pds_ecg2_db[:len(f_ecg2)//10], label = '2')
plt.legend(loc='upper right')
plt.show()

FileNotFoundError: [Errno 2] No such file or directory: 'ecg_sin_ruido.npy'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Como podemos ver, elegí dos curvas, una tiene L=2 y otra L=10. Como la relacion de energías de mi método de Welch es un 8% mayor a la energía de mi ECG, y la resolución se conserva incluso reduciendo el ruido, entonces conservo este valor.

### PPG

In [ ]:
#%% PPG pletismografía
fs_ppg = 400 # Hz
##################
## PPG con ruido
##################
"""
# # Cargar el archivo CSV como un array de NumPy
# ppg = np.genfromtxt('PPG.csv', delimiter=',', skip_header=1)  # Omitir la cabecera si existe
"""
##################
## PPG sin ruido
##################

ppg = np.load('ppg_sin_ruido.npy')

f_ppg, PDS_ppg = sig.welch(ppg, fs_ppg, nperseg = len(ppg)/2)
f_ppg2, PDS_ppg2 = sig.welch(ppg, fs_ppg, nperseg = len(ppg)/15)

# Normalizamos en dB
var_PPG = np.var(ppg)
pds_ppg_db = 10* (np.log10(PDS_ppg)) - 10*np.log10(var_PPG)
pds_ppg2_db = 10* (np.log10(PDS_ppg2)) - 10*np.log10(var_PPG)

# Análisis de Energía
df = f_ppg2[2]-f_ppg2[1]
en_PDS_ppg2 = np.sum(PDS_ppg2)*df
print(en_PDS_ppg2/var_PPG)

plt.figure()
plt.title("PPG Sin Ruido")
plt.xlabel('time [s]')
plt.ylabel('Amplitud [V]')
plt.plot(ppg)
plt.show()

plt.figure()
plt.title("PPG Sin Ruido [dB(Hz)]")
plt.xlabel('frequency [Hz]')
plt.ylabel('PSD [dB]')
plt.plot(f_ppg, pds_ppg_db, label = '1')
plt.plot(f_ppg2, pds_ppg2_db, label = '2')
plt.legend(loc='upper right')
plt.show()


plt.figure()
plt.title("PPG Sin Ruido [dB(Hz)]")
plt.xlabel('frequency [Hz]')
plt.ylabel('PSD [dB]')
plt.plot(f_ppg[:len(f_ppg)//20], pds_ppg_db[:len(f_ppg)//20], label = '1')
plt.plot(f_ppg2[:len(f_ppg2)//20], pds_ppg2_db[:len(f_ppg2)//20], label = '2')
plt.legend(loc='upper right')
plt.show()

La energía se mantiene un 2% por debajo de la energía de la señal del PPG y en la zona más significativa de mi espectro de potencia (0Hz-5Hz) conserva una buena relación, entonces considero este un buen criterio para estimar mi PSD.

### Audio


In [ ]:
#%% Audio
# Cargar el archivo CSV como un array de NumPy
fs_audio, wav_data = sio.wavfile.read('la cucaracha.wav')
# fs_audio, wav_data = sio.wavfile.read('prueba psd.wav')
# fs_audio, wav_data = sio.wavfile.read('silbido.wav')
nn = np.arange(0,len(wav_data))
tt = nn/(fs_audio)

plt.figure()
plt.title("La Cucaracha")
plt.xlabel('time [s]')
plt.ylabel('Amplitud [V]')
plt.plot(tt, wav_data)
#nperseg = cantidad de muestras por segmento (promedio de cuantas muestras. No se mide en muestras si no cantidad de segmentos por audio/archivo)
frecuencias, periodograma = sig.welch(wav_data,fs_audio, nperseg = len(wav_data)/2)
frecuencias2, periodograma2 = sig.welch(wav_data,fs_audio, nperseg = len(wav_data)/15)


periodograma_db = 10* (np.log10(periodograma)) # No va al cuadrado ya que el periodograma ya me da en V**2/Hz
var = np.var(wav_data) # Energia de la señal en el tiempo
periodograma_db = 10* (np.log10(periodograma)) - 10*np.log10(var)
periodograma_db2 = 10* (np.log10(periodograma2)) - 10*np.log10(var)
""" Para llevarla a la potencia que quiera
p = 2
var_p = np.var(np.sqrt(p)*wav_data/np.sqrt(var))
"""
print(var)
df = frecuencias[1]-frecuencias[0]

print(np.sum(periodograma)*df)


plt.figure()
plt.title("E. Periodograma por Welch")
plt.xlabel('frequency [Hz]')
plt.ylabel('PSD [dB]')
plt.plot(frecuencias, periodograma_db, label ="1")
plt.plot(frecuencias2, periodograma_db2,label = "2")
plt.legend(loc='upper right')
plt.show()



# ARREGLAR ESTO -->



plt.figure()
plt.title("E. Periodograma por Welch")
plt.xlabel('frequency [Hz]')
plt.ylabel('PSD [dB]')
plt.plot(frecuencias[len(frecuencias)//100:len(frecuencias)//20], periodograma_db[len(frecuencias)//100:len(frecuencias)//20], label ="1")
plt.plot(frecuencias2[len(frecuencias)//100:len(frecuencias)//20], periodograma_db2[len(frecuencias)//100:len(frecuencias)//20],label = "2")
plt.legend(loc='upper right')
plt.show()

Acá podemos ver que mis frecuencias más significativas llegan hasta los 10kHz, lo cual tiene sentido ya que el rango audible humano y el rango vocal o de los silbidos cae entre los 20 y los 20kHz. Sin embargo, vemos que hay una gran concentración de energía entre los 1000 y los 3000Hz. Veamos qué pasa con los anchos de banda si pedimos que un 98% de la energía de nuestra señal esté dentro de nuestro ancho de banda

In [ ]:
#%% Ancho de Banda
def ancho_de_banda(f, psd, energia_max):
    energia_acum = np.cumsum(psd) # Vector que guarda la info de la energia acumulada hasta el indice
    # Algo tipo: a_n = sumatoria(0,n) vector(n) donde vector(n) es el n-ésimo valor de mi vector "vector" V.L.R
    energia_acum = energia_acum/energia_acum[-1] # La ultima energía es la total, lo estoy normalizando
    f_inf = f[np.searchsorted(energia_acum, (1-energia_max)/2)]
    f_sup = f[np.searchsorted(energia_acum, 1-(1-energia_max)/2)]
    return f_inf, f_sup, f_sup-f_inf

resultados = {} # Armo un diccionario
resultados['ECG'] = ancho_de_banda(f_ecg2, PDS_ecg2, 0.98)
resultados['PPG'] = ancho_de_banda(f_ppg2, PDS_ppg2, 0.98)
resultados['Audio'] = ancho_de_banda(frecuencias2, periodograma2, 0.98)

tablaBW = pd.DataFrame(resultados, index=['f_inf [Hz]','f_sup [Hz]','BW [Hz]']).T # ver el tema de las frecuencias, no sé si está en bines
print(tablaBW)

Acá armé para la la señal de audio el código y le pedí a la IA (Claude) que me genere el mismo proceso para las tres señales y que me haga una tabla.